# Практическое занятие № 15 — аудит достоверности ML-результатов
**ТюмГУ · «Технологии искусственного интеллекта — 1» · 90 минут**

Задача: **аудировать регрессионную модель времени доставки**, а не получать «красивую метрику». Проверяем постановку и признаки, целевую переменную, качество источника, утечки, baseline, ошибки по группам, временную устойчивость и воспроизводимость.

**Важно:** ранее тестовая часть уже могла использоваться в работе № 14; поэтому здесь она **не используется**. Проверяем только `dev = train + validation` во временной кросс-валидации. Это не новый независимый финальный тест. Все числа на `demo_orders_13_14.csv` — демонстрационные.

## 1. Загрузка, очистка и контракт признаков (0–20 мин)
Проверить исходный файл, допустимое время возникновения признаков, правила очистки и количество исключённых записей. Подставьте свой `orders_dirty.csv`; демонстрационный CSV применяется только для отладки.

In [ ]:
# ОБЩАЯ ПОДГОТОВКА: выполняется самостоятельно в каждом блокноте.
# В Colab загрузите orders_dirty.csv через боковую панель «Файлы».
# При отсутствии официального файла используйте только demo_orders_13_14.csv
# и ОТМЕТЬТЕ В ОТЧЁТЕ, что результаты демонстрационные.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = Path('orders_dirty.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('demo_orders_13_14.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Загрузите orders_dirty.csv (учебный источник) либо demo_orders_13_14.csv (только демонстрация).')
source_type = 'ОФИЦИАЛЬНЫЙ/УЧЕБНЫЙ НАБОР' if DATA_PATH.name=='orders_dirty.csv' else 'ДЕМОНСТРАЦИОННЫЕ ДАННЫЕ — НЕ СДАВАТЬ КАК ОСНОВНОЙ ИСТОЧНИК'
print('Источник:', DATA_PATH.name, '|', source_type)
raw = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
REQUIRED = {'order_id','created_at','distance_km','queue_size','courier_load','promised_minutes','zone','weather','actual_minutes','late'}
assert REQUIRED.issubset(raw.columns), f'Отсутствуют поля: {sorted(REQUIRED-set(raw.columns))}'
print('Исходный размер:', raw.shape, 'Полных повторов:', int(raw.duplicated().sum()))

# Для регрессии actual_minutes — цель; late запрещено использовать как признак.
# Пустой/нечисловой target не подходит для fit и оценки; диапазон 5–240 —
# учебное правило качества, согласованное со словарём данных, а не физический закон.
df=raw.drop_duplicates().copy()
df['created_at']=pd.to_datetime(df['created_at'], errors='coerce')
df['actual_minutes']=pd.to_numeric(df['actual_minutes'], errors='coerce')
valid_target=df['actual_minutes'].between(5,240,inclusive='both')
valid_time=df['created_at'].notna()
valid_id=df['order_id'].notna()
print('Цель вне 5–240 или NaN:',int((~valid_target).sum()),
      'невалидных дат:',int((~valid_time).sum()),'пустых ID:',int((~valid_id).sum()))
df=df.loc[valid_target & valid_time & valid_id].copy()
# Если разные строки имеют один ID, студент обязан разобраться с ним до разбиения.
assert df['order_id'].is_unique, 'Конфликтующие order_id: разберите вручную до продолжения.'
df=df.sort_values(['created_at','order_id'],kind='stable').reset_index(drop=True)
assert len(df)>=45, f'Для этой демонстрации нужно хотя бы 45 валидных заказов, найдено {len(df)}'
n=len(df)
n_train=int(n*.6); n_dev=int(n*.8)
train=df.iloc[:n_train].copy(); val=df.iloc[n_train:n_dev].copy(); test=df.iloc[n_dev:].copy()
assert len(train)>0 and len(val)>0 and len(test)>0
assert set(train['order_id']).isdisjoint(val['order_id'])
assert set(train['order_id']).isdisjoint(test['order_id'])
assert set(val['order_id']).isdisjoint(test['order_id'])
assert train['created_at'].max()<=val['created_at'].min()<=test['created_at'].min()
print('После очистки:',n,'train/val/test:',list(map(len,[train,val,test])))
print('Временные диапазоны:')
for name,part in [('train',train),('val',val),('test (не открывать до занятия 14!)',test)]:
    print(name, len(part),str(part.created_at.min()),'…',str(part.created_at.max()))

NUM_BASE=['distance_km','queue_size','courier_load','promised_minutes']
CAT=['zone','weather']
DERIVED=['hour_sin','hour_cos','planned_speed_kmh']
NUM=NUM_BASE+DERIVED
INPUT=NUM_BASE+CAT+['created_at']

def make_features(X):
    # Функция НЕ обучается по данным и применяется идентично внутри каждого fold.
    f=X.loc[:,INPUT].copy()
    for k in NUM_BASE:
        f[k]=pd.to_numeric(f[k], errors='coerce')
        f.loc[f[k]<0,k]=np.nan
    f.loc[f['promised_minutes']<=0,'promised_minutes']=np.nan
    for k in CAT:
        s=f[k].astype('string').str.strip()
        s=s.mask(s.str.lower().isin(['','nan','none','null','нет данных','неизвестно']))
        f[k]=s.astype(object).where(s.notna(),np.nan)
    ts=pd.to_datetime(f['created_at'],errors='coerce')
    hour=ts.dt.hour.astype(float)
    f['hour_sin']=np.sin(2*np.pi*hour/24)
    f['hour_cos']=np.cos(2*np.pi*hour/24)
    f['planned_speed_kmh']=60*f['distance_km']/f['promised_minutes']
    f['planned_speed_kmh']=f['planned_speed_kmh'].replace([np.inf,-np.inf],np.nan)
    return f.drop(columns=['created_at'])

num_pipe=Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),('scale',StandardScaler())])
cat_pipe=Pipeline([('impute',SimpleImputer(strategy='constant',fill_value='неизвестно')),
                   ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
prep=ColumnTransformer([('num',num_pipe,NUM),('cat',cat_pipe,CAT)],remainder='drop')

def build_pipeline(regressor):
    return Pipeline([('derive',FunctionTransformer(make_features,validate=False)),
                     ('prep',clone(prep)),('model',regressor)])

def eval_regression(y_true,y_pred):
    return {'MAE':float(mean_absolute_error(y_true,y_pred)),
            'RMSE':float(np.sqrt(mean_squared_error(y_true,y_pred))),
            'R2':float(r2_score(y_true,y_pred))}
Xtrain=train[INPUT].copy(); ytrain=train['actual_minutes'].copy()
Xval=val[INPUT].copy(); yval=val['actual_minutes'].copy()
# test intentionally saved but not used before final check in 14.
assert {'order_id','late','actual_minutes'}.isdisjoint(Xtrain.columns)
print('Входные поля:',INPUT,'Цель: actual_minutes (мин)')

In [ ]:
# Аудит источника без подглядывания в test-метрики.
raw_a=raw.copy()
raw_a['actual_num']=pd.to_numeric(raw_a['actual_minutes'],errors='coerce')
raw_a['promised_num']=pd.to_numeric(raw_a['promised_minutes'],errors='coerce')
raw_a['late_num']=pd.to_numeric(raw_a['late'],errors='coerce')
check=(raw_a['actual_num']-raw_a['promised_num']>15).astype('Int64')
mask=raw_a[['actual_num','promised_num','late_num']].notna().all(axis=1)
conflicts=int((check[mask]!=raw_a.loc[mask,'late_num']).sum())
checks=pd.DataFrame([
 {'Проверка':'Строк в источнике','Значение':len(raw)},
 {'Проверка':'Полные повторы','Значение':int(raw.duplicated().sum())},
 {'Проверка':'Невалидное время доставки','Значение':int((~raw_a['actual_num'].between(5,240)).sum())},
 {'Проверка':'Конфликт late с правилом > 15 мин (при полных данных)','Значение':conflicts},
 {'Проверка':'Строк в рабочей таблице','Значение':len(df)},
 {'Проверка':'Входные признаки доступны до доставки','Значение':set(INPUT).isdisjoint({'late','actual_minutes','order_id'})},
])
print(checks.to_string(index=False))
checks.to_csv('15_source_audit.csv',index=False,encoding='utf-8-sig')
assert not set(INPUT)&{'late','actual_minutes','order_id'}
# Какова политика по конфликтным late? Для регрессии эта метка не используется,
# но проблема качества исходного источника должна быть задокументирована.

## 2. Временная проверка и базовая модель (20–42 мин)
Сравнить `DummyRegressor(median)` и `Ridge(alpha=10)` на одних и тех же 4 временных фолдах. Считать MAE/RMSE только в проверочных частях. Ни один трансформер не обучается по всей выборке заранее.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

dev=pd.concat([train,val],ignore_index=True).sort_values(['created_at','order_id']).reset_index(drop=True)
Xdev=dev[INPUT].copy(); ydev=dev['actual_minutes'].copy()
cv=TimeSeriesSplit(n_splits=4)
models={'Dummy':DummyRegressor(strategy='median'), 'Ridge':Ridge(alpha=10.0)}
fold_rows=[]; oof_rows=[]
for fold,(i_tr,i_va) in enumerate(cv.split(Xdev),start=1):
    assert dev.iloc[i_tr]['created_at'].max()<=dev.iloc[i_va]['created_at'].min()
    for model_name,est in models.items():
        pipe=build_pipeline(clone(est))
        pipe.fit(Xdev.iloc[i_tr],ydev.iloc[i_tr])
        pred=pipe.predict(Xdev.iloc[i_va])
        mae=float(mean_absolute_error(ydev.iloc[i_va],pred))
        rmse=float(np.sqrt(mean_squared_error(ydev.iloc[i_va],pred)))
        fold_rows.append({'fold':fold,'model':model_name,'n_train':len(i_tr),
                          'n_validation':len(i_va),'MAE_min':mae,'RMSE_min':rmse})
        for ix,actual,p in zip(i_va,ydev.iloc[i_va],pred):
            oof_rows.append({'index':int(ix),'fold':fold,'model':model_name,
                            'order_id':dev.iloc[ix]['order_id'],
                            'created_at':dev.iloc[ix]['created_at'],
                            'zone':dev.iloc[ix]['zone'],'weather':dev.iloc[ix]['weather'],
                            'actual_minutes':float(actual),'prediction':float(p),
                            'residual':float(actual-p),'AE':float(abs(actual-p))})
folds=pd.DataFrame(fold_rows);oof=pd.DataFrame(oof_rows)
print(folds.round(2).to_string(index=False))
summary=folds.groupby('model')[['MAE_min','RMSE_min']].agg(['mean','std']).round(2)
print('\nСредние и вариативность метрик по фолдам:\n',summary)
folds.to_csv('15_fold_scores.csv',index=False,encoding='utf-8-sig')
oof.to_csv('15_oof_predictions.csv',index=False,encoding='utf-8-sig')
assert set(oof['model'])==set(models) and oof['prediction'].notna().all()

## 3. Диагностика по периодам и группам (42–67 мин)
Постройте график MAE по фолдам, распределение остатков и таблицу по зонам. Не сравнивайте группы из 2 и 30 заказов как одинаково надёжные. Остаток `e = факт − прогноз`; положительный означает недооценку времени.

In [ ]:
import matplotlib.pyplot as plt
r=oof[oof['model']=='Ridge'].copy()
pivot=folds.pivot(index='fold',columns='model',values='MAE_min')
ax=pivot.plot(kind='bar',figsize=(8,4));ax.set(xlabel='Временной фолд',ylabel='MAE, мин',title='Временная устойчивость моделей');plt.tight_layout();plt.savefig('15_fold_mae.png',dpi=160);plt.show()
fig,ax=plt.subplots(figsize=(7.5,4));ax.scatter(r['prediction'],r['residual'],alpha=.6,s=24)
ax.axhline(0,ls='--');ax.set(xlabel='Прогноз, мин',ylabel='Факт − прогноз, мин',title='Остатки Ridge (только OOF)');fig.tight_layout();fig.savefig('15_residuals.png',dpi=160);plt.show()
by_zone=r.groupby('zone',dropna=False).agg(n=('AE','size'),MAE_min=('AE','mean'),median_AE=('AE','median')).sort_values('n',ascending=False)
by_zone['достаточная_группа_n>=10']=by_zone['n']>=10
print(by_zone.round(2).to_string());by_zone.to_csv('15_groups.csv',encoding='utf-8-sig')
print('Группы с n < 10 — только описательные наблюдения, без устойчивых выводов.')

## 4. Проверка сдвига признаков и условный вердикт (67–82 мин)
Сравните входы в train и validation **не используя test**. Учебные критерии пригодности — заданные для упражнения правила, а не научно подтверждённые пороги промышленной системы. Сформулируйте минимум три риска и меры контроля.

In [ ]:
# Проверка качества входов на двух последовательных периодах без чтения test.
shift=[]
for feature in ['distance_km','queue_size','promised_minutes','courier_load']:
    a=pd.to_numeric(train[feature],errors='coerce')
    b=pd.to_numeric(val[feature],errors='coerce')
    shift.append({'feature':feature,'median_train':a.median(),'median_validation':b.median(),
                  'missing_train_pct':100*a.isna().mean(),'missing_validation_pct':100*b.isna().mean()})
shift=pd.DataFrame(shift);print(shift.round(2).to_string(index=False))
shift.to_csv('15_feature_shift.csv',index=False,encoding='utf-8-sig')

mae_dummy=float(folds[folds.model=='Dummy'].MAE_min.mean())
mae_ridge=float(folds[folds.model=='Ridge'].MAE_min.mean())
max_fold_ridge=float(folds[folds.model=='Ridge'].MAE_min.max())
# Условный gate: измените его только с объяснением предметной области.
quality_gate={'schema_valid':bool(set(INPUT).isdisjoint({'late','actual_minutes','order_id'})),
 'better_than_baseline':bool(mae_ridge<mae_dummy),
 'all_fold_mae_below_35_min':bool(max_fold_ridge<35.0)}
print('Учебный вердикт:',quality_gate)
print('ВЫВОД:', 'условно проходит' if all(quality_gate.values()) else 'требует доработки')
import json
report={'source':DATA_PATH.name,'scope':'development time-CV only; test not used',
        'cv_folds':4,'mae_dummy_min':mae_dummy,'mae_ridge_min':mae_ridge,
        'quality_gate_example':quality_gate,
        'caveats':['synthetic educational data possible','group sizes and time drift matter','not an independent deployment approval']}
Path('15_model_audit.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')

## 5. Самостоятельная часть и отчёт (82–90 мин)
1. Составьте таблицу «риск → доказательство → возможное искажение → действие» для трёх рисков: leakage, временной сдвиг, малые группы.
2. Сравните качество двух моделей с baseline по средним MAE и разбросу на фолдах.
3. Укажите, почему исторический `test` после работы № 14 нельзя объявлять новым независимым тестом.
4. Дайте краткий вердикт «пригодна / нуждается в доработке / данных недостаточно» со ссылкой на конкретные числа.

**Сдать:** `Группа_Фамилия_З15_Отчет.pdf` (или DOCX) + `15_student.ipynb` + `15_model_audit.json`. Скриншоты **не заменяют** таблицу метрик; блокнот должен выполняться после Restart & Run All.